# ASTR 457: The Midterm

Posted Thu Oct 8, due Wed Oct 14 by Noon (fork → PR, as always). Graded like a lab.

Three parts. Each is designed to test something every lab so far has come back to - which error bar is the right one, whether the model needs the extra parameter, and whether the check you ran could have failed. You've now got the tools to do all of this, so let's do it on numbers you haven't seen: 250 real Type Ia supernovae, and someone else's analysis of real white dwarfs.

Part 1 is 30 pts (6 each), Part 2 is 45 pts, and Part 3 is 25 pts. You'll need 2b and 2c right to get 2d right.

Remember, the rules are the syllabus rules - open book, you can use AI tools (document them in 3b), you can work together and discuss problems and solutions, but you have to write up your own solution and run it on **your own data** in `data/<your netid>/` (nobody else's numbers will match yours).

Put your notebook in `submissions/<your netid>/midterm.ipynb`. The path below (`DATA`) reaches your files from there. Make sure it runs top to bottom from a fresh kernel ("Restart & Run All") - that is how it will be graded, and how it will be run if you defend it.

Throughout, make sure every number you quote has a unit, an error bar and a sensible number of digits (two significant figures on the error bar, and round the value to match).

In [ ]:
#pip install arviz
import matplotlib.style.core
import arviz as az

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

NETID = 'ms155'   # <-- set this
DATA = f'../../labs/midterm/data/{NETID}/'   # from submissions/<your netid>/

def load(name):
    """Read one of your CSV files as a pandas table; d['column'].to_numpy() gives a numpy array. Names stay strings."""
    return pd.read_csv(DATA + name + '.csv', dtype={'name': str})

## Part 1: Error bars (30 pts, 6 each)

Five short scenarios. Each one is a number someone handed you - say whether it's right, compute the right one, and say in a sentence or two why. Show the code. Each of these should be about 20 minutes of work.

### 1a - The extinction coefficient (`p1a_extinction.csv`)

An observer measured a photometric zero point at 15 different airmasses through one night. Extinction in magnitudes grows in proportion to the airmass $X$, so the zero point should follow a line, $m = k\,X + m_0$, where $k$ is the extinction coefficient in mag per airmass. Columns: `airmass`, `zeropoint_mag`, `sigma_quoted_mag` (the pipeline's error on each point).

1. Fit the line by weighted least squares with the quoted errors. Report $k$ and its error bar computed from the quoted errors alone.
2. Now run `np.polyfit(airmass, zeropoint_mag, 1, w=1/sigma_quoted_mag, cov=True)` and report its error bar on $k$. Explain exactly what `polyfit` did to get a different number (read its documentation for `cov`).
3. Compute $\chi^2$ and the number of degrees of freedom. How probable is a $\chi^2$ this large or this small if the quoted errors were right? (Compute it from the $\chi^2$ distribution.)
4. Report the error bar on $k$ you'd quote, and justify it. What does your answer to (3) tell you about the pipeline's error bars?

In [ ]:
# 1a: your work here
p1aextin =load("p1a_extinction")
airmass = np.array(p1aextin["airmass"])
zp_mag = np.array(p1aextin["zeropoint_mag"])
sigma_quoted_mag = np.array(p1aextin["sigma_quoted_mag"])
plt.errorbar(airmass,zp_mag,sigma_quoted_mag,fmt="o",ecolor="grey",capsize=3,label="Data")
N = len(airmass)
ones = np.ones(N)
X = np.column_stack([ones,airmass])                              #O
m_0 , k = np.linalg.inv(X.T @ X) @ (X.T @ zp_mag)                #L
OLSzp = k*airmass + m_0                                          #S
sigma = np.diag(sigma_quoted_mag**2)                                   #W
covariance = np.linalg.inv(X.T @ np.linalg.inv(sigma) @ X)             #L
m_0WLS, kWLS = covariance @ X.T @ np.linalg.inv(sigma) @ zp_mag        #S
WLSzp= kWLS*airmass + m_0WLS                                           #fit
plt.plot(airmass,WLSzp,label="WLS fit")
ksigWLS = np.sqrt(covariance[1][1])
print(f"WLS k = {kWLS} uncertainty WLS ksig = {ksigWLS}")
polyf = np.polyfit(airmass, zp_mag, 1, w=1/sigma_quoted_mag, cov=True)
m_0polyf, kpolyf = polyf[0][1], polyf[0][0]
polyfzp= kpolyf*airmass + m_0polyf                                      
plt.plot(airmass,polyfzp,label="polyfit fit")
plt.xlabel("airmass")
plt.ylabel("zero point magnitude")
plt.title("WLS vs polyfit")
plt.legend()
ksigpolyf = np.sqrt(polyf[1][1][1])
print(f"polyfit k = {kpolyf} uncertainty polyfit ksig = {ksigpolyf}")
chi2 = np.sum(((zp_mag-polyfzp) / sigma_quoted_mag) ** 2)
dof = N-2
redchi2 = chi2/dof
print(f"polyfit reduced chi2 = {redchi2}")
polyunscale = np.polyfit(airmass, zp_mag, 1, w=1/sigma_quoted_mag, cov="unscaled")
ksigunscale = np.sqrt(polyunscale[1][1][1])
print(f"unscaled polyfit error ksig = {ksigunscale}")

**ANSWER:**

*The WLS fit for k is .1311 as extinction coefficient with error ksig = .0097992. The polyfit for k is .1311 and is identical to WLS but its error ksig = .01915. The reason for this difference is that the covariance in polyfit is scaled by the reduced chi squared. Since reduced chi 2 is 1.267, it is still near 1 and the error bars are close to the residuals and could be trusted as reduced chi2 would be around 1 if error bars scaled to the residuals which they seem to do. Since the residuals appear to be comparable to sigma, I believe the error scaling is unnecessary so I believe ksig = .01701 which is polyfits unscaled covariance.*

### 1b - The centroid (`p1b_centroid.csv`)

A classmate measured the position of a star from `n_photons` photons, each landing at a position drawn from a Gaussian of width `psf_sigma_arcsec` around the true position. They report an error on the position of `quoted_sigma_x0_arcsec`, from a clever weighting scheme they found online.

1. Compute the Cramér-Rao bound on the error of any unbiased estimator of the position (Lab 02).
2. Is the quoted error possible? If not, say what the classmate's number must mean instead, and name two concrete mistakes that would produce an error bar this small.

In [ ]:
# 1b: your work here
p1bcentroid =load("p1b_centroid")
psfsigarc = p1bcentroid["psf_sigma_arcsec"][0]
nphoton = p1bcentroid["n_photons"][0]
quotedsigarc = p1bcentroid["quoted_sigma_x0_arcsec"][0]
I = nphoton/psfsigarc**2 # fisher info
variancecrb = 1/I
crbsig = np.sqrt(variancecrb)
print(f"The minimum sigma by cramer rao is {crbsig:.5f} vs the quoted sigma arc {quotedsigarc:.5f}")

**ANSWER:**

*Since Nphoton/psfsigma^2 is the fisher information/likelihood integral, we can get cramer rao bound out of it. The quoted error is smaller than the cramer rao bound so it should not be possible. He could be talking about a biased estimator as cramer rao is strictly unbiased, he could also have expected information skewwed by observed information which could have reduced uncertainty, if we still assume a gaussian psf.*

### 1c - The transit depth's error bar (`p1c_profile.csv`)

A transit-depth fit produced this profile: columns `depth_ppt` (the depth, in parts per thousand) and `minus2lnL` ($-2\ln L$ at that depth, other parameters fixed at their best values). A classmate took the second derivative of `minus2lnL` at its minimum and reported $\sigma = 1/\sqrt{d^2(-2\ln L)/d\,\textrm{depth}^2}$.

1. Find the best depth and the correct $1\sigma$ error from the curvature. Is the classmate's error bar right? If not, by what factor is it off, and why?
2. Read the $\Delta(-2\ln L) = 1$ interval directly off the profile. Is it symmetric? Compare it with your curvature answer.
3. If the classmate quotes $\pm$ their $\sigma$ on many independent datasets, what fraction of those intervals contain the true depth? Predict it, then check it with a simulation (a few thousand fake best-fit depths drawn around a truth you choose, using your correct $\sigma$).

In [ ]:
# 1c: your work here
p1cprofile =load("p1c_profile")
p1cprofile
depthppt = p1cprofile["depth_ppt"]
min2lnL = p1cprofile["minus2lnL"]
Iminlike = np.argmin(min2lnL)
bestdepth = depthppt[Iminlike]
bestmin2lnL = min2lnL[Iminlike] 
print(f"since {(depthppt[Iminlike+1]+depthppt[Iminlike-1])/2} = {bestdepth} we can use h as the difference between the depth suurounding our best depth")
h = depthppt[Iminlike+1]-depthppt[Iminlike]
secdermin2lnL = (min2lnL[Iminlike-1] - 2 * min2lnL[Iminlike] + min2lnL[Iminlike+1])/h**2
classsig = 1/np.sqrt(secdermin2lnL)
hess = secdermin2lnL
covarmin2lnL = 2 / hess # since there is only one term the inverse hessian is just 1/(d2lnL/ddepth2), the 2 is because curature is based on lnL not 2lnL
curvsig = np.sqrt(covarmin2lnL)
print(f"the real curvature error is {curvsig} vs student error {classsig} off by factor of {np.sqrt(2)}")
deltadepthf = np.interp((1+min2lnL[Iminlike]),min2lnL[Iminlike:],depthppt[Iminlike:]) - depthppt[Iminlike]
deltadepthb = depthppt[Iminlike] - np.interp((-min2lnL[Iminlike]-1),-min2lnL[:Iminlike],depthppt[:Iminlike])
print(f"deltadepth forward = {deltadepthf}, deltadepth backward = {deltadepthb}")
print(f"predicting as if gaussian is parabolic that the ratio of true depth is studentsigma**2/curvsigma**2 = {classsig**2/curvsig**2}")
fakedepths = np.random.normal(2,curvsig,5000)
depthratio = len(fakedepths[np.abs(fakedepths-2)<classsig])/5000
print(f"ratio of fake depths within student sigma {depthratio}")

**ANSWER:**

*The curature error = .358 differs from the students answer by a factor of sqrt2 as curvature error is based on -d2(lnL)/ddepth2 or -lnL and not -2lnL meaning the second derivative is off by a factor of 2 so its variance must be 2 times the inverse of d2(-2lnL)/ddepth2 as the -2 comes right out of the derivative leaving the normal expression for variance so the uncertainty would be off by a factor of sqrt2 as sigma = sqrt(variance). Since delta(-2lnL)/deltadepth for delta(-2lnL) = 1 is .3555 in the forward change while -.3586 for behind our minimum -2lnL we can say that the curvature is likely relatively semmetric with our minimum likely slightly offcenter. These values are very similar to .358 our curvature error. I predict gaussian is like a parabolic so the ratio of true depth is studentsigma^2/curvsigma^2 = .5 of the true depths contained in the student error. The answer I got around a true depth of 2 was .5138 meaning it would miss the actual value around half of the time.*

### 1d - The companion mass (`p1d_mass_samples.csv`)

These are 8000 posterior samples of the mass of an unseen companion, `M2_Msun`. Three classmates report its uncertainty as (i) the variance of the samples, (ii) the 84th percentile minus the 16th, and (iii) the standard deviation divided by $\sqrt{8000}$.

1. Report the mass as a median with its 16th-84th percentile error bars, in solar masses.
2. For each of (i)-(iii): what quantity is it actually, what are its units, and by roughly what factor is it wrong as a $1\sigma$ error bar?
3. Is a symmetric $\pm$ error bar appropriate for this posterior? If not, show why.

In [ ]:
# 1d: your work here
p1dmasssamples = load("p1d_mass_samples")
M2Msun = p1dmasssamples["M2_Msun"]
M2median = np.median(M2Msun)
NM2 = len(M2Msun)
M2sort = np.sort(M2Msun)
M2sigvar = np.sum((M2Msun-M2median)**2)/(NM2-1)
M2sig8416 = M2sort[int(NM2*.84-1)] - M2sort[int(NM2*.16-1)]
M2actsig8416 = (M2sort[int(NM2*.84-1)] - M2sort[int(NM2*.16-1)])/2
M2sigSTD = np.sqrt(np.sum((M2Msun-M2median)**2)/(NM2-1))/np.sqrt(NM2)
print(f"median mass = {M2median} and error 16-84 = {M2actsig8416}")
print(f"i./sigma = {M2sigvar/M2actsig8416}, ii./sigma = {M2sig8416/M2actsig8416}, iii.{M2sigSTD/M2actsig8416}")
print(f"i. = {M2sigvar}, ii. = {M2sig8416}, iii = {M2sigSTD}")
print(f"mean of M2 = {np.mean(M2Msun)}")
plt.hist(M2sort,50)
plt.axvline(M2median,c="k",label="Median/predicted mass")
plt.axvspan(M2median-M2actsig8416,M2median+M2actsig8416,color="orange",alpha=.2,label="error from 84th-16th")
plt.xlabel("Mass (M_sun)")
plt.legend()
print(f"ratio below error = {len(M2Msun[M2Msun<M2median-M2actsig8416])/NM2} and ratio above error = {len(M2Msun[M2Msun>M2median+M2actsig8416])/NM2}")

**ANSWER:**

*Reporting the mass as median gives us a mass M2 = 10.8903 (units:Msun) with 16th-84th percentile error as = 2.6757 (units:Msun). If we are to use that as our error, we find i. error = 7.57 with units M_sun^2 with ratio to the actual error of i./sigma = 2.829. ii. error = 5.3514 in units M_sun with ratio off of the actual error ii./sigma = 2. iii. error = .03076 with units M_sun and a ratio to actual error iii./sigma = .0115. Looking at the mean of the masses gives us M_mean = 11.1905 which is ~.3 larger than the median, where true symmetry along the posterior would imply they be near identical. On top of this, looking at the histogram, we see more data to the right of our error than to the left as well as a lot more data behind the median than in front of it within the error bar, showing there exists asymmetry. Quantitatively, the ratio of data below the error vs the total data = .12475 while the ratio above = .1851 or the 81.49th percentile making it asymmetric to the supposed 16th-84th showing the +or- symmetric error as inappropriate.*

### 1e - The slope at fixed intercept (`p1e_slope_intercept_samples.csv`)

Posterior samples of a slope and an intercept from a line fit. A classmate fixed the intercept at its best value, refit only the slope, and quoted the slope's error from that 1-D fit.

1. Compute the slope's error bar the classmate got (the spread of the slope *at fixed intercept*), and the marginal one (the spread of the slope with the intercept marginalized over).
2. Compute their ratio, and the correlation coefficient $\rho$ of the samples. How does the ratio depend on $\rho$? Plot the samples and draw both error bars on the plot.
3. When is the classmate's number the right one to quote? One sentence.

In [ ]:
# 1e: your work here
p1eslopeinterceptsamples = load("p1e_slope_intercept_samples")
p1eslopeinterceptsamples
slope = np.array(p1eslopeinterceptsamples["slope"])
intercept = np.array(p1eslopeinterceptsamples["intercept"])

**ANSWER:**

*your answer goes here*

## Part 2: 250 supernovae and a step (45 pts)

Type Ia supernovae are standardizable candles: their peak brightness, corrected for how fast they fade and for their color, measures distance well enough to have discovered cosmic acceleration. Your file `p2_pantheon_sne.csv` holds 250 real SNe Ia from the Pantheon+ compilation (Scolnic et al. 2022, ApJ 938, 113; [Brout et al. 2022](https://arxiv.org/abs/2202.04077), ApJ 938, 110), all at redshift $0.023 < z < 0.15$. For each supernova, a light-curve fit (SALT2) gives:

- `mB`: the peak apparent magnitude in the B band, and `mB_err`;
- `x1`: the light-curve *stretch* (broader light curves are brighter), and `x1_err`;
- `c`: the *color* (redder supernovae are fainter, from dust and intrinsically), and `c_err`;
- `cov_mB_x1`, `cov_mB_c`, `cov_x1_c`: the covariances between those three, from the same light-curve fit;
- `z`: the redshift; `host_logmass`, `host_logmass_err`: $\log_{10}$ of the host galaxy's stellar mass in solar masses, and its error.

The standardized brightness is the **Tripp relation**,

$$m_B = M + \mu(z) - \alpha\,x_1 + \beta\,c + \gamma\,\Theta(\log M_\star - 10),$$

where $M$ is the absolute magnitude of a "standard" SN Ia, $\mu(z)$ the distance modulus, $\Theta$ is 1 for hosts more massive than $10^{10}\,M_\odot$ and 0 otherwise, and $\gamma$ is the **host-mass step**. Supernovae in massive hosts appear to be brighter after standardization (Kelly et al. 2010; Sullivan et al. 2010); whether the step is real, and why, matters for supernova dark-energy measurements. In this form, a step that makes supernovae in massive hosts brighter has $\gamma < 0$; the published value is about $\gamma \approx -0.06$ mag. So the question for your sample is whether you need $\gamma$ at all.

There are two more things you'll need. At these redshifts the galaxies' own motions (peculiar velocities, about 250 km/s) shift $z$, and so $\mu$; add $\sigma_{\mu,\rm pec} = \frac{5}{\ln 10}\frac{250\ {\rm km/s}}{c\,z}$ to each supernova's variance. And no supernova is a perfect standard candle: add an intrinsic scatter $\sigma_{\rm int}$, a free parameter. Use a flat $\Lambda$CDM distance modulus with $H_0 = 70$ km/s/Mpc and $\Omega_m = 0.3$ (`astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.3).distmod(z)`); $M$ absorbs the choice of $H_0$.

We planted between **two and four contaminants** in your file - objects whose `mB` we made fainter by more than a magnitude, so they are not standard SNe Ia any more (think of a mis-classified peculiar supernova). Their names are not in the file. Real data have a few real outliers too, so not everything far from the line is planted.

### 2a - Look at the data (5 pts)

1. Compute $\mu(z)$ and plot the **Hubble residuals** $m_B - \mu(z)$ against $x_1$, against $c$, and against `host_logmass`.
2. Which way do the $x_1$ and $c$ trends go? Is that the sign the Tripp relation expects? Two or three sentences.
3. The raw trend with host mass may not have the sign of $\gamma$. Why not? (Plot $x_1$ against host mass.)
4. Can you see any of the contaminants?
5. What units do $\alpha$ and $\beta$ have?

In [ ]:
# 2a: your work here
from astropy.cosmology import FlatLambdaCDM
p2pantheonsne = load("p2_pantheon_sne")
z = np.array(p2pantheonsne["z"])
mB = np.array(p2pantheonsne["mB"])
mB_err = np.array(p2pantheonsne["mB_err"])
x1 = np.array(p2pantheonsne["x1"])
x1_err = np.array(p2pantheonsne["x1_err"])
c = np.array(p2pantheonsne["c"])
c_err = np.array(p2pantheonsne["c_err"])
cov_mB_x1 = np.array(p2pantheonsne["cov_mB_x1"])
cov_mB_c = np.array(p2pantheonsne["cov_mB_c"])
cov_x1_c = np.array(p2pantheonsne["cov_x1_c"])
host_logmass = np.array(p2pantheonsne["host_logmass"])
host_logmass_err = np.array(p2pantheonsne["host_logmass_err"])
mu_z = np.array(FlatLambdaCDM(H0=70, Om0=0.3).distmod(z))
hubbleres = mB-mu_z
fig1 = plt.figure()
plt.errorbar(x1,hubbleres,xerr=x1_err,fmt="o",ecolor="grey",capsize=3,label="Data")
plt.xlabel("x_1")
plt.ylabel("hubble residuals (Mag)")
plt.title("hubble residuals vs x_1")
fig2 = plt.figure()
plt.errorbar(c,hubbleres,xerr=c_err,fmt="o",ecolor="grey",capsize=3,label="Data")
plt.xlabel("c")
plt.ylabel("hubble residuals (Mag)")
plt.title("hubble residuals vs c")
fig3 = plt.figure()
plt.errorbar(host_logmass,hubbleres,xerr=host_logmass_err,fmt="o",ecolor="grey",capsize=3,label="Data")
plt.xlabel("host_logmass")
plt.ylabel("hubble residuals (Mag)")
plt.title("hubble residuals vs host_logmass")
fig4 = plt.figure()
plt.errorbar(host_logmass,x1,xerr=host_logmass_err,fmt="o",ecolor="grey",capsize=3,label="Data")
plt.xlabel("host_logmass")
plt.ylabel("x1")
plt.title("x1 vs host_logmass")

**ANSWER:**

*magnitude seems to go down with increasing x1 and go up with increasing c. We can expect this from the tripp relations as increasing hubble residuals imply an increase on the other side where -alpha requires a decreasing x1 to increase and beta requires and increasing c to increase. Looking at the plot for x1 over host_mass, we see that increasing x1 seems to decrease host_logmass which would make since if gamma is negative since alpha is positive, so gamma seems to retain the same sign. There seem to be a few contaminants near the max of hubble residuals that visibly do not fit with the x1 or c plots with them sparse enough on the magnitudes to be seen as the same point. Since x1 has no units, alpha must have units of Mag and since c has units of Mag, beta is dimensionless.*

### 2b - The likelihood (10 pts)

The parameters are $\theta = (M, \alpha, \beta, \gamma, \sigma_{\rm int})$. Each supernova's variance is

$$\sigma_i^2 = \mathbf{v}^T C_i\,\mathbf{v} + \sigma_{\mu,{\rm pec},i}^2 + \sigma_{\rm int}^2, \qquad \mathbf{v} = (1, \alpha, -\beta),$$

where $C_i$ is that supernova's $3\times3$ covariance of $(m_B, x_1, c)$.

1. Write out $\mathbf{v}^T C_i\,\mathbf{v}$ term by term, and explain in one sentence why the off-diagonal terms are there.
2. Write the log-likelihood. Remember the $\ln\sigma_i^2$ term - the variance depends on the parameters.
3. State a prior for every parameter, and justify each in one line.
4. Decide whether to sample $\ln\sigma_{\rm int}$ or $\sigma_{\rm int}$, and say why.
5. Decide what to do about the contaminants. You may model them or remove them, but say how you found them and what a cut would cost you.

In [ ]:
# 2b: your work here
import emcee
covmat = []
for i in range(len(mB_err)):
    covmat.append([[mB_err[i]**2,  cov_mB_x1[i],  cov_mB_c[i]],[cov_mB_x1[i],  x1_err[i]**2,  cov_x1_c[i]],[cov_mB_c[i],  cov_x1_c[i],  c_err[i]**2]])
covmat=np.array(covmat)
def varmat(alpha,beta):
    v = np.array([1,alpha,-beta])
    return v.T @ covmat @ v
btheta = np.ones(len(host_logmass))*(host_logmass -10 > 0)
def loglike(theta):
    M, alpha, beta, gamma, sig_int = theta
    sig_mu = (5/np.log(10)) * 250/(c*z)
    sig_i2 = varmat(alpha,beta) + sig_mu**2 + sig_int**2
    value = M - mB + mu_z - alpha * x1 + beta * c + gamma * btheta * (host_logmass-10)
    return -.5 *np.sum((value)**2/sig_i2 + np.log(sig_i2) + np.log(2*np.pi))
def logprior(theta):
    M, alpha, beta, gamma, sig_int = theta
    if not M<30: # the magnitude is significantly smaller than 30 considering all terms at 0 except host_logmass which ends anyway and hubble residuals
        return -np.inf
    if not 1>alpha >-.5: # alpha seems to be around 1/2 so this well contains it
        return -np.inf
    if not 10>beta >0: # beta seems to be around 5 so this well contains it
        return -np.inf
    if not .5>gamma > -1: # gamma is likely small so this should contain it
        return -np.inf
    if not sig_int >= 0: # it is strictly positive so this should contain it
        return -np.inf
    return 0
def logpost(theta):
    logposts = -(logprior(theta) + loglike(theta))
    if not np.isfinite(logposts):
        return -np.inf
    return logposts

**ANSWER:**

*I placed the off diagonals where cov_mB_x1 is top middle and left middle, cov_mB_c as top right and bottom left, and cov_x1_c as bottom middle and right middle since the matrix would iterate from mB to x1 to c as the v vector would need to multiply mB variance by 1, x1 variance by alpha and c variance by -beta. I chose to leave sig_int without log as i am unsure what to place my prior over besides positive and it could very well be in the realm of 0. You can find the contaminants in the fainter magnitudes above the data by comparing both the hubble residuals to x1 and c and finding points that fail to fit both. I chose to ultimately ignore them as I believed it would not significantly alter the result.*

### 2c - Sampling and convergence (12 pts)

1. Find the maximum a posteriori point. Then run **four independent chains** from starting points spread wider than the posterior (you choose how, and say how you chose). Any sampler is fine - your own Metropolis, `emcee` (one ensemble per chain), or `PyMC`.
2. Report $\hat R$ and the effective sample size for every parameter (`arviz.rhat`, `arviz.ess`, or `arviz.summary`), and say what values would have made you run longer.
3. Make a corner plot, and quote $\alpha$, $\beta$, $\gamma$ and $\sigma_{\rm int}$ as medians with 16th-84th percentile error bars.
4. Which parameters are correlated strongly? Comment on why. Then fix $M$ at its MAP value, refit $\gamma$, and compare that $\sigma_\gamma$ with the marginal one (cf. 1e).

`ArviZ` 1.x (install it with `conda install -c conda-forge pymc arviz`, as in `INSTALL.txt`) takes your chains as a dictionary of arrays shaped (chain, draw): `idata = az.from_dict({'posterior': {'alpha': a, 'beta': b}})`, then `az.summary(idata)`.

In [ ]:
# 2c: your work here
def negloglike(theta):
    return -loglike(theta)
def neglogpost(theta):
    return -logpost(theta)
P0 = [-100,.5,5,-.06,.1]
minimspost = []
minimsparam =[]
for sig_int in np.linspace(0,1,10,endpoint=False):
    for gamma_0 in np.linspace(-.5,.5,endpoint=False):
        for M_0 in np.linspace(-100,20,10,endpoint=False):
            #M_0 = P0[0]
            #gamma_0 = P0[3]
            sig_int0 = P0[4]
            alpha_0 = P0[1]
            beta_0 = P0[2]
            
            theta_0 = np.array([M_0,alpha_0,beta_0,gamma_0,sig_int0])
            minims = minimize(neglogpost,theta_0,method='Nelder-Mead')
            if minims.success:
                minimspost.append(logpost(minims.x))
                minimsparam.append(minims.x)


In [ ]:
minimsparam

In [ ]:
ndim = 5
nwalkers = 32                      
nsteps = 5000
nchains = 4
rng = np.random.default_rng(10052026)
scale = 1e-3 * np.abs(theta0)
chains = []
for k in range(nchains):
    rng = np.random.default_rng(1000 + k)
    p0 = theta0 + scale * rng.standard_normal((nwalkers, ndim))
    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior)
    sampler.random_state = rng.bit_generator.state   
    sampler.run_mcmc(p0, nsteps, progress=True)
    chains.append(sampler)

**ANSWER:**

*I first fit a maximum posterior to get initial values for the emcee and then*

### 2d - Model comparison (10 pts)

Compare $M_0$ ($\gamma = 0$) with $M_1$ ($\gamma$ free).

1. Compute $2\ln B_{1,0}$ with the Laplace approximation (Day 13), with a prior on $\gamma$ you state and defend. How does it change if that prior is twice as wide?
2. Compute $\textrm{BIC}_0 - \textrm{BIC}_1$ for comparison.
3. How strong is your answer on Day 8's scale (0-2 weak, 2-6 positive, 6-10 strong, above 10 very strong)? Compare $\gamma$ with the published $-0.06$ mag. Could your 250 supernovae have detected a step that size? (One way: your $\sigma_\gamma$ tells you how many $\sigma$ a $-0.06$ step would be.)

In [ ]:
# 2d: your work here

**ANSWER:**

*your answer goes here*

### 2e - Testing your pipeline (8 pts)

Design one test of your pipeline that would come out differently if your pipeline were wrong, and run it. State **before** you run it what result would count as a failure. One suggestion: generate fake supernovae at your redshifts, stretches and colors from parameters you choose (including a $\gamma$ you choose), fit 30 or more such datasets (maximum-likelihood fits with curvature error bars are fine here; you don't need MCMC for each), and check how often your 68% intervals contain the truth.

In [ ]:
# 2e: your work here

**ANSWER:**

*your answer goes here*

## Part 3: The referee report (25 pts)

White dwarfs are the cores of dead stars like the Sun. Your file `p3_wd_masses.csv` holds 600 real DA (hydrogen-atmosphere) white dwarfs from the 100 pc sample of [Kilic et al. (2020)](https://arxiv.org/abs/2006.00323), ApJ 898, 84: `name`, `teff_K`, `mass_Msun`, `mass_err_Msun`. Most single white dwarfs have masses near $0.6\,M_\odot$; Kilic et al. report a narrow peak there plus an excess of more massive ones, around $0.7$-$0.9\,M_\odot$, which merger products alone do not explain.

The notebook `labs/midterm/ai_solution.ipynb` fits your masses with a two-population model using `emcee`. Run it from that directory, after setting `NETID` in its first cell. It runs top to bottom and the plots look nice, but it's wrong in at least three substantive ways - and if you don't look at your data, you'll fool yourself the same way.

### 3a - The report (20 pts)

1. **Identify** at least three substantive errors (not style; "it used 32 walkers and I'd use 48" is not one).
2. **Demonstrate** each on *your* file: show, with a number or a plot, what the error does to the answer. A demonstration can come out negative: if an error doesn't move your numbers, show that, and say what it would take for it to matter.
3. **State** the correct treatment, and run it where you can.

In [ ]:
# 3a: your work here
# 1st mistake
p3wdmasses = load("p3_wd_masses")
p3wdmasses
mdmass = p3wdmasses["mass_Msun"]
mdmasserr = p3wdmasses["mass_err_Msun"]
temp = p3wdmasses["teff_K"]
fig1 = plt.figure()
plt.hist(mdmass, 50)
wdmedian = np.median(mdmass)
wdSTD = np.std(mdmass)
plt.axvline(wdmedian,c="k",label="median")
plt.axvspan(wdmedian-3*wdSTD,wdmedian + 3*wdSTD,color="orange",alpha=.2)
fig2 = plt.figure()


# ai pipeline section
m = mdmass
sd = wdSTD
med = wdmedian
def log_prob(p, m):
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < 0.5 and 0.005 < s2 < 0.5 and 0 < f < 1):
        return -np.inf
    return np.sum(np.log(f * stats.norm.pdf(m, mu1, s1) + (1 - f) * stats.norm.pdf(m, mu2, s2)))

# starting guess from a quick split of the sample at 0.7 Msun
lo, hi = m[m < 0.7], m[m >= 0.7]
p0 = np.array([lo.mean(), lo.std(), hi.mean(), hi.std(), lo.size / m.size])
np.random.seed(42)
nwalk, ndim = 32, 5
start = p0 + 1e-3 * np.random.randn(nwalk, ndim)
sampler = emcee.EnsembleSampler(nwalk, ndim, log_prob, args=(m,))
sampler.run_mcmc(start, 3000, progress=False)
chain = sampler.get_chain(discard=500)              # (steps, walkers, params)
flat = chain.reshape(-1, ndim)
print(f"mean acceptance fraction: {np.mean(sampler.acceptance_fraction):.2f}")
def gelman_rubin(x):                                  # x: (steps, chains)
    n, mchains = x.shape
    B = n * np.var(x.mean(axis=0), ddof=1)
    W = np.mean(np.var(x, axis=0, ddof=1))
    return np.sqrt(((n - 1) / n * W + B / n) / W)

names = ['mu1', 'sigma1', 'mu2', 'sigma2', 'f']
for i, nm in enumerate(names):
    print(f"R-hat({nm}) = {gelman_rubin(chain[:, :, i]):.3f}")
print("All R-hat < 1.1: the chains have converged.")

fig, axs = plt.subplots(5, 1, figsize=(8, 8), sharex=True)
for i, nm in enumerate(names):
    axs[i].plot(chain[:, :, i], lw=0.3, alpha=0.5); axs[i].set_ylabel(nm)
axs[-1].set_xlabel('step'); plt.tight_layout(); plt.show()
best = np.median(flat, axis=0)
rng = np.random.default_rng(1)
n2 = rng.binomial(m.size, 1 - best[4])
sim = np.concatenate([rng.normal(best[0], best[1], m.size - n2), rng.normal(best[2], best[3], n2)])
vs = emcee.EnsembleSampler(nwalk, ndim, log_prob, args=(sim,))
vs.run_mcmc(best + 1e-3 * np.random.randn(nwalk, ndim), 2000, progress=False)
rec = np.median(vs.get_chain(discard=500, flat=True), axis=0)
for nm, a, b in zip(names, best, rec):
    print(f"{nm}: input {a:.3f}, recovered {b:.3f} ({100 * (b - a) / a:+.1f}%)")
print("Input parameters recovered: the pipeline is validated.")
mu1, s1, mu2, s2, f = best
print(f"Population 1: mean mass {mu1:.3f} +/- {s1:.3f} Msun ({100 * f:.0f}% of white dwarfs)")
print(f"Population 2: mean mass {mu2:.3f} +/- {s2:.3f} Msun ({100 * (1 - f):.0f}% of white dwarfs)")

one = stats.norm.fit(m)
lnL1 = np.sum(stats.norm.logpdf(m, *one)); lnL2 = log_prob(best, m)
print(f"log-likelihood: one population {lnL1:.1f}, two populations {lnL2:.1f}; improvement {lnL2 - lnL1:.1f}")

x = np.linspace(m.min(), m.max(), 300)
plt.hist(m, 50, density=True, color='0.8', label='white dwarfs')
plt.plot(x, f * stats.norm.pdf(x, mu1, s1), label='population 1')
plt.plot(x, (1 - f) * stats.norm.pdf(x, mu2, s2), label='population 2')
plt.plot(x, f * stats.norm.pdf(x, mu1, s1) + (1 - f) * stats.norm.pdf(x, mu2, s2), 'k', label='total')
plt.xlabel('mass [Msun]'); plt.legend(); plt.show()

In [ ]:

# 2nd mistake, entire pipeline with only log p changed
mass = mdmass
sd = wdSTD
med = wdmedian
keep = np.abs(mass - med) < 3 * sd
m = mass[keep]
def log_prob(p, m):
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < 0.5 and 0.005 < s2 < 0.5 and 0 < f < 1):
        return -np.inf
    sig1 = np.sqrt(s1**2+mdmasserr[keep]**2)
    sig2 = np.sqrt(s2**2+mdmasserr[keep]**2)
    return np.sum(np.log(f * stats.norm.pdf(m, mu1, sig1) + (1 - f) * stats.norm.pdf(m, mu2, sig2)))

# starting guess from a quick split of the sample at 0.7 Msun
lo, hi = m[m < 0.7], m[m >= 0.7]
p0 = np.array([lo.mean(), lo.std(), hi.mean(), hi.std(), lo.size / m.size])
np.random.seed(42)
nwalk, ndim = 32, 5
start = p0 + 1e-3 * np.random.randn(nwalk, ndim)
sampler = emcee.EnsembleSampler(nwalk, ndim, log_prob, args=(m,))
sampler.run_mcmc(start, 3000, progress=False)
chain = sampler.get_chain(discard=500)              # (steps, walkers, params)
flat = chain.reshape(-1, ndim)
print(f"mean acceptance fraction: {np.mean(sampler.acceptance_fraction):.2f}")
def gelman_rubin(x):                                  # x: (steps, chains)
    n, mchains = x.shape
    B = n * np.var(x.mean(axis=0), ddof=1)
    W = np.mean(np.var(x, axis=0, ddof=1))
    return np.sqrt(((n - 1) / n * W + B / n) / W)

names = ['mu1', 'sigma1', 'mu2', 'sigma2', 'f']
for i, nm in enumerate(names):
    print(f"R-hat({nm}) = {gelman_rubin(chain[:, :, i]):.3f}")
print("All R-hat < 1.1: the chains have converged.")

fig, axs = plt.subplots(5, 1, figsize=(8, 8), sharex=True)
for i, nm in enumerate(names):
    axs[i].plot(chain[:, :, i], lw=0.3, alpha=0.5); axs[i].set_ylabel(nm)
axs[-1].set_xlabel('step'); plt.tight_layout(); plt.show()
best = np.median(flat, axis=0)
rng = np.random.default_rng(1)
n2 = rng.binomial(m.size, 1 - best[4])
sim = np.concatenate([rng.normal(best[0], best[1], m.size - n2), rng.normal(best[2], best[3], n2)])
vs = emcee.EnsembleSampler(nwalk, ndim, log_prob, args=(sim,))
vs.run_mcmc(best + 1e-3 * np.random.randn(nwalk, ndim), 2000, progress=False)
rec = np.median(vs.get_chain(discard=500, flat=True), axis=0)
for nm, a, b in zip(names, best, rec):
    print(f"{nm}: input {a:.3f}, recovered {b:.3f} ({100 * (b - a) / a:+.1f}%)")
print("Input parameters recovered: the pipeline is validated.")
mu1, s1, mu2, s2, f = best
print(f"Population 1: mean mass {mu1:.3f} +/- {s1:.3f} Msun ({100 * f:.0f}% of white dwarfs)")
print(f"Population 2: mean mass {mu2:.3f} +/- {s2:.3f} Msun ({100 * (1 - f):.0f}% of white dwarfs)")

one = stats.norm.fit(m)
lnL1 = np.sum(stats.norm.logpdf(m, *one)); lnL2 = log_prob(best, m)
print(f"log-likelihood: one population {lnL1:.1f}, two populations {lnL2:.1f}; improvement {lnL2 - lnL1:.1f}")

x = np.linspace(m.min(), m.max(), 300)
plt.hist(m, 50, density=True, color='0.8', label='white dwarfs')
plt.plot(x, f * stats.norm.pdf(x, mu1, s1), label='population 1')
plt.plot(x, (1 - f) * stats.norm.pdf(x, mu2, s2), label='population 2')
plt.plot(x, f * stats.norm.pdf(x, mu1, s1) + (1 - f) * stats.norm.pdf(x, mu2, s2), 'k', label='total')
plt.xlabel('mass [Msun]'); plt.legend(); plt.show()

In [ ]:
# 3rd mistake, entire pipeline with only -log p minimized for starting values
mass = mdmass
sd = wdSTD
med = wdmedian
keep = np.abs(mass - med) < 3 * sd
m = mass[keep]
def log_prob(p, m):
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < 0.5 and 0.005 < s2 < 0.5 and 0 < f < 1):
        return -np.inf
    return np.sum(np.log(f * stats.norm.pdf(m, mu1, s1) + (1 - f) * stats.norm.pdf(m, mu2, s2)))

# starting guess from a quick split of the sample at 0.7 Msun
#def neglog_prob(p, m):
    #return -log_prob(p,m)
lo, hi = m[m < 0.7], m[m >= 0.7]
p0 = np.array([lo.mean(), lo.std(), hi.mean(), hi.std(), lo.size / m.size])
#p0 = minimize(neglog_prob,p0,args=(m),method="Nelder-Mead").x
np.random.seed(42)
nwalk, ndim = 32, 5
start = p0 + p0*1e-3 * np.random.randn(nwalk, ndim)
sampler = emcee.EnsembleSampler(nwalk, ndim, log_prob, args=(m,))
sampler.run_mcmc(start, 3000, progress=False)
chain = sampler.get_chain(discard=500)              # (steps, walkers, params)
flat = chain.reshape(-1, ndim)
print(f"mean acceptance fraction: {np.mean(sampler.acceptance_fraction):.2f}")
def gelman_rubin(x):                                  # x: (steps, chains)
    n, mchains = x.shape
    B = n * np.var(x.mean(axis=0), ddof=1)
    W = np.mean(np.var(x, axis=0, ddof=1))
    return np.sqrt(((n - 1) / n * W + B / n) / W)

names = ['mu1', 'sigma1', 'mu2', 'sigma2', 'f']
for i, nm in enumerate(names):
    print(f"R-hat({nm}) = {gelman_rubin(chain[:, :, i]):.3f}")
print("All R-hat < 1.1: the chains have converged.")

fig, axs = plt.subplots(5, 1, figsize=(8, 8), sharex=True)
for i, nm in enumerate(names):
    axs[i].plot(chain[:, :, i], lw=0.3, alpha=0.5); axs[i].set_ylabel(nm)
axs[-1].set_xlabel('step'); plt.tight_layout(); plt.show()
best = np.median(flat, axis=0)
rng = np.random.default_rng(1)
n2 = rng.binomial(m.size, 1 - best[4])
sim = np.concatenate([rng.normal(best[0], best[1], m.size - n2), rng.normal(best[2], best[3], n2)])
vs = emcee.EnsembleSampler(nwalk, ndim, log_prob, args=(sim,))
vs.run_mcmc(best + 1e-3 * np.random.randn(nwalk, ndim), 2000, progress=False)
rec = np.median(vs.get_chain(discard=500, flat=True), axis=0)
for nm, a, b in zip(names, best, rec):
    print(f"{nm}: input {a:.3f}, recovered {b:.3f} ({100 * (b - a) / a:+.1f}%)")
print("Input parameters recovered: the pipeline is validated.")
mu1, s1, mu2, s2, f = best
print(f"Population 1: mean mass {mu1:.3f} +/- {s1:.3f} Msun ({100 * f:.0f}% of white dwarfs)")
print(f"Population 2: mean mass {mu2:.3f} +/- {s2:.3f} Msun ({100 * (1 - f):.0f}% of white dwarfs)")

one = stats.norm.fit(m)
lnL1 = np.sum(stats.norm.logpdf(m, *one)); lnL2 = log_prob(best, m)
print(f"log-likelihood: one population {lnL1:.1f}, two populations {lnL2:.1f}; improvement {lnL2 - lnL1:.1f}")

x = np.linspace(m.min(), m.max(), 300)
plt.hist(m, 50, density=True, color='0.8', label='white dwarfs')
plt.plot(x, f * stats.norm.pdf(x, mu1, s1), label='population 1')
plt.plot(x, (1 - f) * stats.norm.pdf(x, mu2, s2), label='population 2')
plt.plot(x, f * stats.norm.pdf(x, mu1, s1) + (1 - f) * stats.norm.pdf(x, mu2, s2), 'k', label='total')
plt.xlabel('mass [Msun]'); plt.legend(); plt.show()
print(f"p0 = {p0}")

**REFEREE REPORT:**

*The report started off by cleaning using a 3 sigma clip with standard deviation. It did so before considering if it was even necessessary. In this case we can see that not only is the median likely skewwed to the right by a large amount of data that appears to be the second gaussian. This is not symmetric and clipping the data also fails to consider the second gaussian so the correct treatment is to not make the clip. Taking the pipeline of the unclipped data gives mu1 going from .584 to .585, sigma1 from .024 to .026, mu2 from .699 to .717, sigma2 from .172 to .196, f from .334 to .357 and a negative change to log likelihood. While error and likelihood were hurt by this decision, its more authentic and mu2 was changed significantly. Second, the log probability did not consider error between individual stars important or significant and ignored them. Changing only the log probability to consider it resulted in mu1 going from .584 to .583, sigma1 from .024 to .019, mu2 from .699 to .702, sigma2 unchanged, f from .334 to .355 and a 1.2 improvement to log likelihood. There is a significant positive difference in those results in both sigma1 and f for including individual errors. As for the third mistake, because walkers iterate around the initial values where the values themselves are not the same and can vary in scale, the walk you add must compare with their size. Instead of using p0 + .001 random normal we can use p0(1+.001random normal). In this case only sigma2 changed from .172 to .171, and f changed from .334 to .337 but if one of the parameters was closer to .001 in value, the difference would be much greater as it would struggle to converge.*

### 3b - AI-use appendix (5 pts)

1. Which tools did you use, for what, what did they get wrong, and how did you catch it? "I used ChatGPT to help" is useless; "none" is fine if that's true. I promise I don't care which - just be honest about what you did.
2. What would have gone wrong in your answer if you had not checked the AI's output? Name the specific thing, in two or three sentences.

**AI-USE APPENDIX:**

*1. I asked claude "how do i find covariance" as a reminder on how to do so. I used it as a guide and checked online and in lecture just to remind myself how exactly I had to find covariance. If I hadn't checked elsewhere my entire covariance matrix would be off. I also asked for clarification on log likelihood vs log probability which also helped clarify. If i hadn't checked online and in lecture, my probability estimate would be off. I asked claude to help write me code for emcee walkers with 4 independent chains which I only took chunks of code and looked over them so my emcee would not be wrong*

Get started. You know how to do all of this. If not, there's something in a lab or class notebook that can help. Slack me with questions. Best of luck.